# 02 - Feature Engineering
Builds the analysis-ready feature set from the cleaned data: bucketed
numeric fields, a quarter-of-year seasonal flag, and behavioral flags
derived from prior-campaign history. Logic lives in
`src/feature_engineering.py` so the EDA, SQL, R, and modeling assets all
consume an identical, versioned definition.

In [1]:
import sys
sys.path.append('..')
import pandas as pd

from src.feature_engineering import run as run_features, CATEGORICAL_FEATURES, NUMERIC_REALISTIC, NUMERIC_BENCHMARK

feat_df = run_features(save=True)
print(feat_df.shape)
feat_df.head()

(41176, 27)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,cons_conf_idx,euribor3m,nr_employed,y,age_group,campaign_bucket,quarter,contacted_before,prior_success,pdays_bucket
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,0,56-65,1,Q2,0,0,never_contacted
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,0,56-65,1,Q2,0,0,never_contacted
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,-36.4,4.857,5191.0,0,36-45,1,Q2,0,0,never_contacted
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,-36.4,4.857,5191.0,0,36-45,1,Q2,0,0,never_contacted
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,-36.4,4.857,5191.0,0,56-65,1,Q2,0,0,never_contacted


## Engineered columns

In [2]:
feat_df[['age', 'age_group', 'campaign', 'campaign_bucket',
         'pdays', 'pdays_bucket', 'month', 'quarter',
         'previous', 'contacted_before', 'poutcome', 'prior_success']].sample(10, random_state=1)

,age,age_group,campaign,campaign_bucket,pdays,pdays_bucket,month,quarter,previous,contacted_before,poutcome,prior_success
39117,26,26-35,1,1,5,0-7_days,dec,Q4,1,1,success,1
17910,25,18-25,6,4-6,999,never_contacted,jul,Q3,0,0,nonexistent,0
29436,52,46-55,1,1,999,never_contacted,apr,Q2,0,0,nonexistent,0
13843,42,36-45,1,1,999,never_contacted,jul,Q3,0,0,nonexistent,0
35290,36,36-45,3,2-3,999,never_contacted,may,Q2,0,0,nonexistent,0
8284,31,26-35,1,1,999,never_contacted,jun,Q2,0,0,nonexistent,0
26962,33,26-35,2,2-3,999,never_contacted,nov,Q4,0,0,nonexistent,0
34848,24,18-25,5,4-6,999,never_contacted,may,Q2,0,0,nonexistent,0
18146,51,46-55,3,2-3,999,never_contacted,jul,Q3,0,0,nonexistent,0
17170,39,36-45,2,2-3,999,never_contacted,jul,Q3,0,0,nonexistent,0


## Why these features
- **age_group**: raw age is noisy for a marketing segment; 6 life-stage
  bins map directly onto campaign personas.
- **campaign_bucket**: the number of contacts this campaign has strong
  diminishing (and eventually negative) returns - bucketing exposes that
  non-linearity to linear models and to a Power BI slicer.
- **pdays_bucket**: `pdays=999` is a sentinel meaning "never contacted",
  not a numeric distance of 999 days - leaving it untouched would badly
  mislead any model or chart that treats it as a magnitude.
- **quarter**: coarser seasonality than month, useful for a compact
  dashboard filter and for R's ANOVA test in `r/statistical_analysis.R`.
- **contacted_before / prior_success**: collapse `previous` and
  `poutcome` into the two behavioral signals that actually matter for
  targeting - has this client ever responded before?

## Feature sets used downstream

In [3]:
print('Categorical features:', CATEGORICAL_FEATURES)
print()
print('Numeric (realistic - no duration):', NUMERIC_REALISTIC)
print()
print('Numeric (benchmark - includes duration):', NUMERIC_BENCHMARK)

Categorical features: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome', 'age_group', 'campaign_bucket', 'pdays_bucket', 'quarter']

Numeric (realistic - no duration): ['age', 'campaign', 'previous', 'contacted_before', 'prior_success', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed']

Numeric (benchmark - includes duration): ['age', 'campaign', 'previous', 'contacted_before', 'prior_success', 'emp_var_rate', 'cons_price_idx', 'cons_conf_idx', 'euribor3m', 'nr_employed', 'duration']


In [4]:
feat_df.to_csv('../data/processed/bank_marketing_features.csv', index=False)
print('Saved data/processed/bank_marketing_features.csv:', feat_df.shape)

Saved data/processed/bank_marketing_features.csv: (41176, 27)
